# CareCost Fusion — Snowflake × Google Vertex AI

**Gemini proposes. XGBoost proves. Snowflake governs.**

A hybrid healthcare-cost demo: Gemini (on Vertex, via function calling) proposes features,
the Vertex Gen AI Evaluation Service scores them, and a deterministic XGBoost holdout gate —
not the model — accepts or rejects each one. Snowflake governs the data.

> **Live only — no fallbacks.** Requires `GOOGLE_CLOUD_PROJECT` + ADC and a configured
> `~/.snowflake/connections.toml`. Cells raise clear errors if a service is missing.
> Synthetic data, no PHI.

## 1 · Setup & authentication

In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np, pandas as pd, yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
try:
    from dotenv import load_dotenv; load_dotenv(ROOT / ".env")
except Exception:
    pass

cfg_path = ROOT / "config.yaml"
if not cfg_path.exists():
    cfg_path = ROOT / "config.example.yaml"
CONFIG = yaml.safe_load(cfg_path.read_text())
DATA_CFG, MODEL_CFG, EVAL_CFG, GEMINI_CFG = (CONFIG[k] for k in ("data", "model", "evaluation", "gemini"))

PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]          # required — no fallback
LOCATION = os.getenv("GOOGLE_CLOUD_LOCATION", "us-central1")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
BUCKET = f"{PROJECT}-carecost"
print("Vertex project:", PROJECT, "| region:", LOCATION)

In [ ]:
# Snowflake source — requires an allowlisted IP (turn off any VPN exit node that changes it).
from snowflake_io import get_connection
conn = get_connection()
print("Snowflake:", conn.cursor().execute("SELECT CURRENT_WAREHOUSE(), CURRENT_DATABASE(), CURRENT_SCHEMA()").fetchone())

## 2 · Build features in Snowflake\n\nLoads synthetic claims and builds `MEMBER_FEATURES_BASE` in-warehouse (SQL time-windows).

In [ ]:
import pipeline as P
from features import BASELINE_MODEL_FEATURES
features_df = P.features_from_snowflake(conn, member_count=DATA_CFG["member_count"], seed=DATA_CFG["random_seed"])
print(f"{len(features_df):,} feature rows | {features_df.MEMBER_ID.nunique():,} members")
features_df.head()

## 3 · Baseline vs. median → Vertex AI Experiments

In [ ]:
from vertex_experiments import ExperimentLogger
base = P.train_baseline(features_df)
logger = ExperimentLogger("carecost-fusion", PROJECT, LOCATION, staging_bucket=f"gs://{BUCKET}")
logger.log_run("median-baseline", {"model_type": "MEDIAN"}, base.median_metrics)
logger.log_run("xgb-baseline", {"model_type": "XGB_BASELINE", "features": len(BASELINE_MODEL_FEATURES),
                                "n_estimators": MODEL_CFG["n_estimators"], "max_depth": MODEL_CFG["max_depth"]}, base.metrics)
print(f"median MAE ${base.median_metrics['mae']:,.0f}  |  XGBoost MAE ${base.metrics['mae']:,.0f}  "
      f"|  high-cost recall {base.metrics['high_cost_recall']:.3f}")
print("Experiments:", logger.console_url())

## 4 · Predictions + residual segment → Snowflake\n\nOnly the **aggregate** segment summary is sent to Gemini — no member rows, no IDs.

In [ ]:
from residuals import build_predictions_frame
from snowflake_io import write_df
pred_frame = build_predictions_frame(base.split.test, base.pred, "xgb-baseline", "XGB_BASELINE")
write_df(conn, pred_frame, "MODEL_PREDICTIONS")

segment = P.residual_segment(base)
seg_row = pd.DataFrame([{
    "RUN_ID": "xgb-baseline", "SEGMENT_ID": segment["segment_id"],
    "SEGMENT_DESCRIPTION": segment["segment_description"], "MEMBER_COUNT": segment["member_count"],
    "MEAN_ACTUAL_COST": segment["mean_actual_cost"], "MEAN_PREDICTED_COST": segment["mean_predicted_cost"],
    "MEAN_RESIDUAL": segment["mean_residual"], "MEAN_ABSOLUTE_ERROR": float(np.abs(base.residual).mean()),
    "SEGMENT_EVIDENCE": json.dumps(segment)}])
write_df(conn, seg_row, "RESIDUAL_SUMMARY")
print(json.dumps({k: segment[k] for k in ("segment_description", "member_count", "mean_residual", "conditions")}, indent=2))

## 5 · Gemini on Vertex — function calling + Gen AI Evaluation\n\nGemini calls a `propose_feature` tool bound to the whitelist; each hypothesis is scored by the Gen AI Evaluation Service.

In [ ]:
g = P.gemini_step(segment, project=PROJECT, location=LOCATION, model=GEMINI_MODEL, available_columns=features_df.columns)
print("Gen AI Eval:", g["eval_scores"].get("source"), "| accepted:", g["accepted"])
pd.DataFrame([{"tool_call": f"propose_feature({c.feature_name})", "confidence": c.confidence,
               "plausibility": g["eval_scores"].get(c.feature_name), "hypothesis": c.hypothesis}
              for c in g["response"].candidates])

## 6 · Challengers + deterministic gate → Snowflake\n\nOne challenger per accepted feature; the gate — not Gemini — decides.

In [ ]:
from feature_catalog import CATALOG
hyp = {c.feature_name: c.hypothesis for c in g["response"].candidates}
results = P.challenger_step(features_df, base, g["accepted"], hyp_by_name=hyp)

exp_rows = [{"RUN_ID": "xgb-baseline", "PARENT_RUN_ID": None, "RUN_TYPE": "BASELINE", "FEATURE_NAME": None,
             "FEATURE_SPEC": None, "MAE": base.metrics["mae"], "RMSE": base.metrics["rmse"],
             "RMSLE": base.metrics["rmsle"], "HIGH_COST_RECALL": base.metrics["high_cost_recall"],
             "MAE_IMPROVEMENT_PCT": 0.0, "DECISION": "BASELINE", "DECISION_REASON": "reference", "GEMINI_HYPOTHESIS": None}]
for r in results:
    exp_rows.append({"RUN_ID": r["run_id"], "PARENT_RUN_ID": "xgb-baseline", "RUN_TYPE": "CHALLENGER",
                     "FEATURE_NAME": r["feature_name"], "FEATURE_SPEC": CATALOG[r["feature_name"]].sql_expr,
                     "MAE": r["metrics"]["mae"], "RMSE": r["metrics"]["rmse"], "RMSLE": r["metrics"]["rmsle"],
                     "HIGH_COST_RECALL": r["metrics"]["high_cost_recall"], "MAE_IMPROVEMENT_PCT": r["mae_improvement_pct"],
                     "DECISION": r["decision"], "DECISION_REASON": r["decision_reason"], "GEMINI_HYPOTHESIS": r["hypothesis"]})
    logger.log_run(r["run_id"], {"model_type": "XGB_CHALLENGER", "feature_name": r["feature_name"]},
                   {**r["metrics"], "mae_improvement_pct": r["mae_improvement_pct"]})
write_df(conn, pd.DataFrame(exp_rows), "EXPERIMENT_RESULTS")
for r in results:
    print(f"  {r['feature_name']:24s} {r['mae_improvement_pct']:+.2f}%  ->  {r['decision']}")

## 7 · Register the champion in Vertex Model Registry\n\nOnly the KB model artifact leaves Snowflake; the data never does.

In [ ]:
from modeling import make_model, temporal_split, TARGET
from feature_catalog import materialize
from vertex_registry import register_model
winners = [r for r in results if r["decision"] == "ACCEPT"]
champ_feat = winners[0]["feature_name"] if winners else None
champ_features = BASELINE_MODEL_FEATURES + ([champ_feat] if champ_feat else [])
fdf = features_df.copy()
if champ_feat:
    fdf[champ_feat] = materialize(fdf, champ_feat)
csplit = temporal_split(fdf)
champ = make_model(MODEL_CFG)
champ.fit(csplit.train[champ_features], np.log1p(csplit.train[TARGET].to_numpy()))
model_obj = register_model(champ, display_name="carecost-notebook-champion", bucket=BUCKET, project=PROJECT,
                           metrics=base.metrics, feature_names=champ_features, decision=champ_feat or "baseline")
print("registered:", model_obj.resource_name, "| champion feature:", champ_feat)

## 8 · Final analytical query + conclusions

In [ ]:
from snowflake_io import read_table
final = read_table(conn, '''
    SELECT RUN_ID, FEATURE_NAME, MAE, HIGH_COST_RECALL, MAE_IMPROVEMENT_PCT, DECISION, DECISION_REASON
    FROM EXPERIMENT_RESULTS ORDER BY CREATED_AT''')
display(final.round(3))
conn.close()
print()
print("Snowflake governed the data and computed the features. Vertex AI managed Gemini,")
print("the Gen AI evaluation, experiment tracking, and the model registry. XGBoost — not the")
print("language model — decided each feature on holdout MAE.")